# Test MLflow Logging Frequency with SB3

This notebook tests different `log_interval` values to understand when training metrics are logged.


In [ ]:
import mlflow
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.logger import configure
import sys
sys.path.append('..')
from src.rl_trading_lab.utils.mlflow_logger import MLflowOutputFormat

## Test 1: log_interval=10 (Current Setting)

With `n_steps=2048`, this should log every 10 rollouts = 20,480 steps

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("test-log-interval-10")

with mlflow.start_run(run_name="log_interval_10"):
    env = make_vec_env("CartPole-v1", n_envs=1)
    eval_env = make_vec_env("CartPole-v1", n_envs=1)
    
    model = PPO("MlpPolicy", env, n_steps=2048, verbose=1)
    
    # Setup logger with MLflow
    logger = configure(None, ["stdout"])
    logger.output_formats.append(MLflowOutputFormat())
    model.set_logger(logger)
    
    # Eval every 5000 steps
    eval_callback = EvalCallback(eval_env, eval_freq=5000, n_eval_episodes=3, verbose=1)
    
    print("\n=== Training with log_interval=10 ===")
    print(f"n_steps={model.n_steps}")
    print(f"Expected training log every: {10 * model.n_steps} steps")
    print(f"Eval every: 5000 steps\n")
    
    model.learn(total_timesteps=50000, callback=eval_callback, log_interval=10)
    
    run_id = mlflow.active_run().info.run_id
    print(f"\nRun ID: {run_id}")

In [ ]:
# Check what was logged
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)
metrics = run.data.metrics

print("\n=== Metrics logged (log_interval=10) ===")
for key in sorted(metrics.keys()):
    print(f"  {key}")

# Get metric history for rollout/ep_rew_mean
if 'rollout/ep_rew_mean' in metrics:
    history = client.get_metric_history(run_id, 'rollout/ep_rew_mean')
    steps = [h.step for h in history]
    print(f"\nrollout/ep_rew_mean logged at steps: {steps}")
    print(f"Frequency: every {steps[1] - steps[0] if len(steps) > 1 else 'N/A'} steps")

# Get metric history for eval/mean_reward
if 'eval/mean_reward' in metrics:
    history = client.get_metric_history(run_id, 'eval/mean_reward')
    steps = [h.step for h in history]
    print(f"\neval/mean_reward logged at steps: {steps}")
    print(f"Frequency: every {steps[1] - steps[0] if len(steps) > 1 else 'N/A'} steps")

## Test 2: log_interval=1 (Proposed Fix)

With `n_steps=2048`, this should log every 1 rollout = 2,048 steps

In [ ]:
mlflow.set_experiment("test-log-interval-1")

with mlflow.start_run(run_name="log_interval_1"):
    env = make_vec_env("CartPole-v1", n_envs=1)
    eval_env = make_vec_env("CartPole-v1", n_envs=1)
    
    model = PPO("MlpPolicy", env, n_steps=2048, verbose=1)
    
    # Setup logger with MLflow
    logger = configure(None, ["stdout"])
    logger.output_formats.append(MLflowOutputFormat())
    model.set_logger(logger)
    
    # Eval every 5000 steps
    eval_callback = EvalCallback(eval_env, eval_freq=5000, n_eval_episodes=3, verbose=1)
    
    print("\n=== Training with log_interval=1 ===")
    print(f"n_steps={model.n_steps}")
    print(f"Expected training log every: {1 * model.n_steps} steps")
    print(f"Eval every: 5000 steps\n")
    
    model.learn(total_timesteps=50000, callback=eval_callback, log_interval=1)
    
    run_id = mlflow.active_run().info.run_id
    print(f"\nRun ID: {run_id}")

In [ ]:
# Check what was logged
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)
metrics = run.data.metrics

print("\n=== Metrics logged (log_interval=1) ===")
for key in sorted(metrics.keys()):
    print(f"  {key}")

# Get metric history for rollout/ep_rew_mean
if 'rollout/ep_rew_mean' in metrics:
    history = client.get_metric_history(run_id, 'rollout/ep_rew_mean')
    steps = [h.step for h in history]
    print(f"\nrollout/ep_rew_mean logged at steps: {steps}")
    print(f"Frequency: every ~{(steps[-1] - steps[0]) / (len(steps) - 1) if len(steps) > 1 else 'N/A':.0f} steps")

# Get metric history for eval/mean_reward
if 'eval/mean_reward' in metrics:
    history = client.get_metric_history(run_id, 'eval/mean_reward')
    steps = [h.step for h in history]
    print(f"\neval/mean_reward logged at steps: {steps}")
    print(f"Frequency: every {steps[1] - steps[0] if len(steps) > 1 else 'N/A'} steps")

## Test 3: log_interval=2 (Middle Ground)

With `n_steps=2048`, this should log every 2 rollouts = 4,096 steps

In [ ]:
mlflow.set_experiment("test-log-interval-2")

with mlflow.start_run(run_name="log_interval_2"):
    env = make_vec_env("CartPole-v1", n_envs=1)
    eval_env = make_vec_env("CartPole-v1", n_envs=1)
    
    model = PPO("MlpPolicy", env, n_steps=2048, verbose=1)
    
    # Setup logger with MLflow
    logger = configure(None, ["stdout"])
    logger.output_formats.append(MLflowOutputFormat())
    model.set_logger(logger)
    
    # Eval every 5000 steps
    eval_callback = EvalCallback(eval_env, eval_freq=5000, n_eval_episodes=3, verbose=1)
    
    print("\n=== Training with log_interval=2 ===")
    print(f"n_steps={model.n_steps}")
    print(f"Expected training log every: {2 * model.n_steps} steps")
    print(f"Eval every: 5000 steps\n")
    
    model.learn(total_timesteps=50000, callback=eval_callback, log_interval=2)
    
    run_id = mlflow.active_run().info.run_id
    print(f"\nRun ID: {run_id}")

In [ ]:
# Check what was logged
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)
metrics = run.data.metrics

print("\n=== Metrics logged (log_interval=2) ===")
for key in sorted(metrics.keys()):
    print(f"  {key}")

# Get metric history for rollout/ep_rew_mean
if 'rollout/ep_rew_mean' in metrics:
    history = client.get_metric_history(run_id, 'rollout/ep_rew_mean')
    steps = [h.step for h in history]
    print(f"\nrollout/ep_rew_mean logged at steps: {steps}")
    print(f"Frequency: every {steps[1] - steps[0] if len(steps) > 1 else 'N/A'} steps")

# Get metric history for eval/mean_reward
if 'eval/mean_reward' in metrics:
    history = client.get_metric_history(run_id, 'eval/mean_reward')
    steps = [h.step for h in history]
    print(f"\neval/mean_reward logged at steps: {steps}")
    print(f"Frequency: every {steps[1] - steps[0] if len(steps) > 1 else 'N/A'} steps")

## Summary

Compare the three approaches:
- **log_interval=10**: Training metrics every ~20k steps, eval every 5k steps
- **log_interval=2**: Training metrics every ~4k steps, eval every 5k steps  
- **log_interval=1**: Training metrics every ~2k steps, eval every 5k steps

**Recommendation**: Use `log_interval=2` or `log_interval=1` to get training metrics closer to eval frequency.